In [5]:
import numpy as np
import sys

# --- 1. Define Network Parameters ---
N = 4000
z = 16.24
q_serial = "0000"

# Box geometry
V = 4000.0 
DIM = 3
prefactor = -1.0 / (V * DIM)

# --- 2. Load the C Data ---
grad_file = f"pressureGradient_N{N}_z{z:.2f}_s{q_serial}.dat"
phes_file = f"pressureHessianOnVNA_N{N}_z{z:.2f}_s{q_serial}.dat"
vna_file  = f"vna_N{N}_z{z:.2f}_s{q_serial}.dat" 

try:
    pressure_gradient = np.loadtxt(grad_file).flatten()
    pressure_hessian_on_vna = np.loadtxt(phes_file).flatten()
    
    # CRITICAL FIX 1: Invert sign for relaxation (vna moves opposite to force)
    vna_vector = -1.0 * np.loadtxt(vna_file).flatten()
    
    # --- 3. Compute the Raw Contractions (Reversing the Scaling Bug) ---
    bugged_grad_dot_vna = np.dot(pressure_gradient, vna_vector)
    bugged_vna_dot_phes = np.dot(vna_vector, pressure_hessian_on_vna)

    # CRITICAL FIX 2: Restore the correct exponents to completely remove the C-side scaling
    # Eq 19 has 2 prefactors (one in grad, one in vna)
    raw_grad_dot_vna = bugged_grad_dot_vna / (prefactor ** 2)
    
    # Eq 20 has 4 prefactors (one in vna, three baked into phes by the C code)
    raw_vna_dot_phes = bugged_vna_dot_phes / (prefactor ** 4)

    print(f"Raw Eq 19 Sum (First Order Contraction):  {raw_grad_dot_vna:.6f}")
    print(f"Raw Eq 20 Sum (Second Order Contraction): {raw_vna_dot_phes:.6f}\n")

    # --- 4. Final Thermodynamic Assembly ---
    # Apply your exact analytical prefactors from the alpha theory derivation
    dp_dT_nonaffine = prefactor * raw_grad_dot_vna
    second_derivative = prefactor * raw_vna_dot_phes

    print("=== NON-AFFINE MACROSCOPIC PROPERTIES ===")
    print(f"First Derivative (dp/dT_na): {dp_dT_nonaffine:.6f}")
    print(f"Second Derivative:           {second_derivative:.6f}")

except FileNotFoundError as e:
    print(f"CRITICAL ERROR: {e}")
    sys.exit(1)

Raw Eq 19 Sum (First Order Contraction):  -4182.468564
Raw Eq 20 Sum (Second Order Contraction): -1487.153517

=== NON-AFFINE MACROSCOPIC PROPERTIES ===
First Derivative (dp/dT_na): 0.348539
Second Derivative:           0.123929
